In [7]:
import os
import json

# Thiết lập thông số
os.environ['KAGGLE_USERNAME'] = "KGAT"
os.environ['KAGGLE_KEY'] = "6c065dd23a357c4f8688aee80ccc5205"

# Tạo file cấu hình hệ thống
!mkdir -p ~/.kaggle
data = {"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(data, f)

!chmod 600 ~/.kaggle/kaggle.json
print("Cấu hình xong! Đang thử tải dữ liệu...")

# Tải trực tiếp
!kaggle datasets download -d bhavikjikadara/dog-and-cat-classification-dataset

Cấu hình xong! Đang thử tải dữ liệu...
Dataset URL: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset
License(s): apache-2.0
 91% 702M/775M [00:03<00:01, 49.2MB/s]
100% 775M/775M [00:04<00:00, 202MB/s] 


In [9]:
# Giải nén vào thư mục tên là 'my_data' cho dễ tìm
!unzip -q dog-and-cat-classification-dataset.zip -d ./my_data

# Lệnh này sẽ tự động tìm đường dẫn đến thư mục PetImages dù nó nằm ở đâu
import os
train_root = ""
for root, dirs, files in os.walk('./my_data'):
    if 'PetImages' in dirs:
        train_root = os.path.join(root, 'PetImages')
        break

if train_root:
    print(f"THÀNH CÔNG! Đường dẫn của bạn là: {train_root}")
    # Kiểm tra xem có ảnh không
    print("Các thư mục con:", os.listdir(train_root))
else:
    print(" Vẫn không thấy thư mục. Có thể Username 'KGAT' bị sai, bạn hãy kiểm tra lại tên đăng nhập trên Kaggle nhé.")

THÀNH CÔNG! Đường dẫn của bạn là: ./my_data/PetImages
Các thư mục con: ['Cat', 'Dog']


In [10]:
import os
from PIL import Image

def clean_data(root):
    removed = 0
    for subfolder in ['Cat', 'Dog']:
        folder_path = os.path.join(root, subfolder)
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                with Image.open(file_path) as img:
                    img.verify()
            except:
                os.remove(file_path)
                removed += 1
    print(f"Đã dọn dẹp {removed} file ảnh lỗi.")

clean_data(train_root)

Đã dọn dẹp 0 file ảnh lỗi.


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [11]:
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# Inception v3 yêu cầu size 299x299
train_tf = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_ds = datasets.ImageFolder(train_root, transform=train_tf)

# Chia Train/Val (80/20)
indices = list(range(len(full_ds)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=full_ds.targets)

train_ds = Subset(full_ds, train_idx)
val_ds = Subset(full_ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

print(f"Dữ liệu sẵn sàng: {len(train_ds)} ảnh train, {len(val_ds)} ảnh val.")

Dữ liệu sẵn sàng: 19998 ảnh train, 5000 ảnh val.


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)

# Đóng băng các lớp cũ để train cho nhanh
for param in model.parameters():
    param.requires_grad = False

# Thay thế lớp phân loại cuối cùng cho 2 lớp (Chó/Mèo)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 2)
model.AuxLogits.fc = torch.nn.Linear(model.AuxLogits.fc.in_features, 2) # Inception cần cái này

model = model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 168MB/s] 


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

# 1. Transform: Cực kỳ quan trọng để đạt > 90%
train_tf = transforms.Compose([
    transforms.Resize((128, 128)), # Giảm size xuống 128 để chạy nhanh hơn Inception
    transforms.RandomHorizontalFlip(), # Lật ảnh ngẫu nhiên
    transforms.RandomRotation(15),     # Xoay ảnh nhẹ
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

val_tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Nạp dữ liệu (train_root bạn đã tìm thấy ở các bước trước)
full_ds = datasets.ImageFolder(train_root)

# Chia tách Train/Val
indices = list(range(len(full_ds)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=full_ds.targets, random_state=42)

train_ds = Subset(datasets.ImageFolder(train_root, transform=train_tf), train_idx)
val_ds = Subset(datasets.ImageFolder(train_root, transform=val_tf), val_idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print(f"Data OK! Train: {len(train_ds)} - Val: {len(val_ds)}")

Data OK! Train: 19998 - Val: 5000


In [18]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Layer 1: Tìm nét cơ bản
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # Layer 2: Tìm hình khối
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # Layer 3: Tìm chi tiết mắt, mũi
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        # Layer 4: Phân tích sâu
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Lớp kết nối đầy đủ (Classifier)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5), # Chống học vẹt
            nn.Linear(512, 2) # Đầu ra: Chó hoặc Mèo
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv4): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchN

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Đánh giá sau mỗi epoch
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()

        acc = correct / len(val_ds)
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(train_loader):.4f} - Val Acc: {acc*100:.2f}%")

        if acc > 0.90:
            print("Đã đạt mục tiêu > 90% Accuracy! Dừng sớm để tiết kiệm thời gian.")
            break

train(epochs=15)

Epoch [1/15] - Loss: 0.9252 - Val Acc: 70.70%
Epoch [2/15] - Loss: 0.5538 - Val Acc: 73.78%
Epoch [3/15] - Loss: 0.4966 - Val Acc: 78.98%
Epoch [4/15] - Loss: 0.4476 - Val Acc: 83.52%
Epoch [5/15] - Loss: 0.3984 - Val Acc: 85.66%
Epoch [6/15] - Loss: 0.3639 - Val Acc: 80.70%
Epoch [7/15] - Loss: 0.3328 - Val Acc: 82.56%
Epoch [8/15] - Loss: 0.2995 - Val Acc: 81.90%
Epoch [9/15] - Loss: 0.2966 - Val Acc: 91.02%
Đã đạt mục tiêu > 90% Accuracy! Dừng sớm để tiết kiệm thời gian.


In [ ]:
# Lưu trọng số model
torch.save(model.state_dict(), 'dog_cat_cnn.pth')
print("Đã lưu model thành công!")

# Tải file về máy tính cá nhân
from google.colab import files
files.download('dog_cat_cnn.pth')